# LLM Application in Python
Construct and validate a grounded portfolio explanation.

## 1. Load the portfolio

In [ ]:
import json
from pathlib import Path

portfolio = json.loads(Path("data/sample_portfolio.json").read_text())
print(portfolio)

## 2. Calculate value in Python

In [ ]:
invested = sum(item["shares"] * item["price"] for item in portfolio["holdings"])
total_value = portfolio["cash"] + invested
print(total_value)

## 3. Start a prompt

In [ ]:
question = "Explain this portfolio in two sentences."
prompt = f"Question: {question}"
print(prompt)

## 4. Add JSON context

In [ ]:
context = json.dumps(portfolio, indent=2)
prompt = f"{prompt}\n\nPortfolio facts:\n{context}"
print(prompt[:300])

## 5. Require grounding

In [ ]:
prompt += "\n\nUse supplied facts only. Return JSON with summary and total_value."
print(prompt[-120:])

## 6. Read optional configuration

In [ ]:
import os

endpoint = os.getenv("MODEL_ENDPOINT")
api_key = os.getenv("MODEL_API_KEY")
model_name = os.getenv("MODEL_NAME")
offline = os.getenv("COURSEWARE_OFFLINE", "0") == "1"
print("Configured:", bool(endpoint and api_key and model_name), "Offline:", offline)

## 7. Call or use an offline example

In [ ]:
example_response = json.dumps({
    "summary": "The portfolio combines three synthetic holdings with cash.",
    "total_value": total_value,
})

if endpoint and api_key and model_name and not offline:
    from openai import OpenAI
    client = OpenAI(base_url=endpoint, api_key=api_key)
    raw_response = client.chat.completions.create(
        model=model_name, messages=[{"role": "user", "content": prompt}]
    ).choices[0].message.content
else:
    raw_response = example_response
    print("Offline example response")

## 8. Parse JSON

In [ ]:
parsed = json.loads(raw_response)
print(parsed)

## 9. Validate model output

In [ ]:
from pydantic import BaseModel

class PortfolioExplanation(BaseModel):
    summary: str
    total_value: float

explanation = PortfolioExplanation.model_validate(parsed)
print(explanation)

## 10. Deliberate failure: invalid JSON

In [ ]:
try:
    json.loads("This is not JSON")
except json.JSONDecodeError as error:
    print("Invalid JSON:", error.msg)

## 11. Final grounded result

In [ ]:
assert explanation.total_value == total_value
print(explanation.summary)
print(f"Python-calculated total: ${explanation.total_value:.2f}")

## Why split the work?
Python calculates balances deterministically from supplied numbers. The model is limited to explaining those calculated facts, while parsing and validation check the shape of its response.

## Takeaways
- Build prompts from explicit context.
- Keep arithmetic in Python.
- Parse and validate model output before using it.